In [ ]:
import sys, importlib
!pip install "figuard>=0.3.0" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

budget = client.create_budget(
    user_id="agent_001",
    total_limit=1_000,
    unit="api_calls",
    expires_in="24h",
    authorization_expiry_seconds=60,
    intent_context="web scraping session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit:   {budget.total_limit:,} API calls")

# Extract session token from tokens array (one entry per dimension)
tokens = {t.category: t.session_token for t in budget.tokens}
session_token = tokens["default"]  # simple budget uses "default" category

In [ ]:
# Batch of API calls — authorized
auth = client.authorize(
    session_token=session_token,
    agent_id="scraper_agent",
    action_type="EXTERNAL_CALL",
    description="Serper: batch search queries",
    requested_quantity=50,
    idempotency_key="batch-001",
)
print(f"Batch 1: {auth.decision} — {auth.approved_quantity} calls")
client.confirm_event(auth.event_id, confirmed_quantity=48)
print("✓ Confirmed. 48 calls consumed.")

# Second batch
auth2 = client.authorize(
    session_token=session_token,
    agent_id="scraper_agent",
    action_type="EXTERNAL_CALL",
    description="SerpAPI: image search",
    requested_quantity=200,
    idempotency_key="batch-002",
)
print(f"Batch 2: {auth2.decision} — {auth2.approved_quantity} calls")
client.confirm_event(auth2.event_id, confirmed_quantity=200)
print("✓ Confirmed. 200 calls consumed.")

# Over-limit request
auth3 = client.authorize(
    session_token=session_token,
    agent_id="scraper_agent",
    action_type="EXTERNAL_CALL",
    description="Full site crawl",
    requested_quantity=800,
    idempotency_key="batch-003",
)
print(f"Full crawl: {auth3.decision} — {auth3.denial_reason}")
print("Budget enforced. No calls consumed.")

print(f"\n→ See the spend tree: https://figuard-sandbox-1.onrender.com/ui")